In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from pathlib import Path

# ============================================================================
# LOAD DATA
# ============================================================================

# Choose your version
# Get script location and build path from there
script_dir = Path.cwd()
data_path = script_dir / '..' / 'data' / 'processed' / 'model_ready' / 'features_no_ytd.csv'

print("Loading from:", data_path.resolve())
df = pd.read_csv(data_path)

print(f"Original shape: {df.shape}")
print(f"Target distribution:\n{df['won'].value_counts()}\n")

# ============================================================================
# 1. DROP NON-PREDICTIVE COLUMNS
# ============================================================================

drop_cols = [
    'player_name',      # Names not useful for prediction
    'opponent_name',
    'stats_url',        # Metadata
    'Date',             # Will use time-based split instead
    'ATP',              # Redundant with other context
    'Tournament',       # High cardinality, can cause overfitting
    'Location'          # High cardinality
]

# Keep only columns that exist
drop_cols = [col for col in drop_cols if col in df.columns]
df_model = df.drop(columns=drop_cols)

print(f"After dropping non-predictive columns: {df_model.shape}")

# ============================================================================
# 2. IDENTIFY COLUMN TYPES
# ============================================================================

# Target
target = 'won'

# Categorical columns
categorical_cols = [
    'Surface',          # Hard, Clay, Grass
    'Court',            # Indoor, Outdoor
    'Round',            # Round of 32, Finals, etc.
    'Series',           # ATP250, ATP500, etc.
    'player_hand',      # L, R
    'opponent_hand',    # L, R
]

# Keep only existing categorical columns
categorical_cols = [col for col in categorical_cols if col in df_model.columns]

# Numerical columns (everything else except target)
numerical_cols = [col for col in df_model.columns 
                  if col not in categorical_cols and col != target]

print(f"\nCategorical columns ({len(categorical_cols)}): {categorical_cols}")
print(f"Numerical columns ({len(numerical_cols)}): {len(numerical_cols)} columns")

# ============================================================================
# 3. HANDLE MISSING VALUES
# ============================================================================

print("\n" + "="*80)
print("MISSING VALUES")
print("="*80)
df_model['player_hand'] = df['player_hand'].replace('U','Unknown')
df_model['opponent_hand'] = df['opponent_hand'].replace('U','Unknown')
df_model['Round'] = df['Round'].replace('Round Robin','4th Round')
df_model['Series'] = df['Series'].replace('Masters Cup','ATP500')

missing_summary = df_model.isnull().sum()
missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)
print(f"\nColumns with missing values: {len(missing_summary)}")
for col, count in missing_summary.head(10).items():
    pct = count / len(df_model) * 100
    print(f"  {col:30s}: {count:6d} ({pct:5.1f}%)")

# Strategy for numerical columns
print("\nFilling numerical missing values with median...")
for col in numerical_cols:
    if df_model[col].isnull().any():
        median_val = df_model[col].median()
        df_model[col] = df_model[col].fillna(median_val)

# Strategy for categorical columns
print("Filling categorical missing values with 'Unknown'...")
for col in categorical_cols:
    if df_model[col].isnull().any():
        df_model[col] = df_model[col].fillna('Unknown')

# Verify no missing values remain
remaining_missing = df_model.isnull().sum().sum()
print(f"\nRemaining missing values: {remaining_missing}")
assert remaining_missing == 0, "Still have missing values!"

# ============================================================================
# 4. ENCODE CATEGORICAL VARIABLES
# ============================================================================

print("\n" + "="*80)
print("ENCODING CATEGORICAL VARIABLES")
print("="*80)

# One-hot encode categorical variables
df_model = pd.get_dummies(df_model, columns=categorical_cols, drop_first=True)

print(f"\nShape after encoding: {df_model.shape}")
print(f"Added {df_model.shape[1] - df_model.shape[1] + len(categorical_cols)} dummy variables")

# ============================================================================
# 5. FEATURE ENGINEERING (Optional but recommended)
# ============================================================================

print("\n" + "="*80)
print("FEATURE ENGINEERING")
print("="*80)

# Ranking difference (negative = player is lower ranked / worse)
if 'player_rank' in df_model.columns and 'opponent_rank' in df_model.columns:
    df_model['rank_diff'] = df_model['player_rank'] - df_model['opponent_rank']
    print("Created: rank_diff")

# Elo difference (positive = player has higher Elo / better)
if 'player_elo' in df_model.columns and 'opponent_elo' in df_model.columns:
    df_model['elo_diff'] = df_model['player_elo'] - df_model['opponent_elo']
    print("Created: elo_diff")

# H2H win rate (avoid division by zero)
if 'player_h2h' in df_model.columns and 'opponent_h2h' in df_model.columns:
    total_h2h = df_model['player_h2h'] + df_model['opponent_h2h']
    df_model['player_h2h_rate'] = np.where(
        total_h2h > 0,
        df_model['player_h2h'] / total_h2h,
        0.5  # No history = 50/50
    )
    print("Created: player_h2h_rate")
    
# Rating difference
if 'player_rating' in df_model.columns and 'opponent_rating' in df_model.columns:
    df_model['rating_diff'] = df_model['player_rating'] - df_model['opponent_rating']
    print("Created: rating_diff")
    
# Height difference
if 'player_height' in df_model.columns and 'opponent_height' in df_model.columns:
    df_model['height_diff'] = df_model['player_height'] - df_model['opponent_height']
    print("Created: height_diff")
    
# Point difference
if 'player_points' in df_model.columns and 'opponent_points' in df_model.columns:
    df_model['points_diff'] = df_model['player_points'] - df_model['opponent_points']
    print("Created: points_diff")

# Age difference
if 'player_age' in df_model.columns and 'opponent_age' in df_model.columns:
    df_model['age_diff'] = df_model['player_age'] - df_model['opponent_age']
    print("Created: age_diff")
    
# Try different scale factors
if 'player_rank' in df_model.columns and 'opponent_rank' in df_model.columns:
    df_model['proba_rank_200'] = 1 / (1 + 10 ** ((df_model['player_rank'] - df_model['opponent_rank']) / 200))
    print("Created: proba_rank_200")
    
# Different scale factor for form
if 'player_form' in df_model.columns and 'opponent_form' in df_model.columns:
    df_model['proba_form_50'] = 1 / (1 + 10 ** ((df_model['player_form'] - df_model['opponent_form']) / 50))
    print("Created: proba_form_50")

print(f"\nFinal shape with engineered features: {df_model.shape}")

# ============================================================================
# 6. SEPARATE FEATURES AND TARGET
# ============================================================================

X = df_model.drop(columns=[target])
y = df_model[target]

print("\n" + "="*80)
print("TRAIN-TEST SPLIT")
print("="*80)

# Option A: Random split (simple)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Train target distribution: {y_train.value_counts().to_dict()}")
print(f"Test target distribution: {y_test.value_counts().to_dict()}")

# ============================================================================
# 7. SCALE NUMERICAL FEATURES (Important for some models)
# ============================================================================

print("\n" + "="*80)
print("FEATURE SCALING")
print("="*80)

# Identify numerical columns to scale
cols_to_scale = [col for col in X.columns if not any(
    cat in col for cat in categorical_cols
)]

# Identify probability columns (already 0-1 scale)
proba_cols = [col for col in cols_to_scale if 'proba_' in col or col == 'player_win_prob_elo']

# OPTION A: Scale everything including probabilities (DEFAULT)
cols_to_scale_final = cols_to_scale
print(f"\nUsing OPTION A: Scaling ALL {len(cols_to_scale_final)} columns")
print("  (Probabilities will be scaled too)")

# OPTION B: Exclude probabilities from scaling (UNCOMMENT TO USE)
# cols_to_scale_final = [col for col in cols_to_scale if col not in proba_cols]
# print(f"\nUsing OPTION B: Scaling {len(cols_to_scale_final)} columns")
# print(f"  (Excluding {len(proba_cols)} probability columns)")

# Apply scaling
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[cols_to_scale_final] = scaler.fit_transform(X_train[cols_to_scale_final])
X_test_scaled[cols_to_scale_final] = scaler.transform(X_test[cols_to_scale_final])

print("\nScaling complete.")
# ============================================================================
# 8. SUMMARY
# ============================================================================

print("\n" + "="*80)
print("PREPROCESSING COMPLETE")
print("="*80)

print(f"\nFinal dataset:")
print(f"  Total samples: {len(X):,}")
print(f"  Total features: {X.shape[1]:,}")
print(f"  Train samples: {len(X_train):,}")
print(f"  Test samples: {len(X_test):,}")
print(f"  Target balance: {y.mean():.1%} won")

print("\nReady for modeling!")
print("\nYou now have:")
print("  X_train, X_test, y_train, y_test - Original scale")
print("  X_train_scaled, X_test_scaled - Scaled features")
print("\nUse original scale for tree-based models (Random Forest, XGBoost)")
print("Use scaled features for linear models (Logistic Regression, SVM)")

Loading from: C:\Users\harry\Desktop\tennis_betting_efficiency-features\data\processed\model_ready\features_no_ytd.csv
Original shape: (49750, 33)
Target distribution:
won
1    24875
0    24875
Name: count, dtype: int64

After dropping non-predictive columns: (49750, 27)

Categorical columns (6): ['Surface', 'Court', 'Round', 'Series', 'player_hand', 'opponent_hand']
Numerical columns (20): 20 columns

MISSING VALUES

Columns with missing values: 16
  opponent_rating               :  12670 ( 25.5%)
  player_rating                 :  12670 ( 25.5%)
  player_height                 :   4020 (  8.1%)
  opponent_height               :   4020 (  8.1%)
  opponent_hand                 :   3996 (  8.0%)
  player_hand                   :   3996 (  8.0%)
  player_age                    :   3996 (  8.0%)
  opponent_age                  :   3996 (  8.0%)
  form_diff                     :   3270 (  6.6%)
  player_form                   :   1798 (  3.6%)

Filling numerical missing values with median.

In [3]:
from sklearn.metrics import log_loss, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix


def eval_classifier(y_true, y_proba, threshold=0.5):
    """
    y_proba: predicted probability for class 1
    """
    y_pred = (y_proba >= threshold).astype(int)
    
    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_proba)
    ll = log_loss(y_true, y_proba)
    mae = mean_absolute_error(y_true, y_proba)
    r2 = r2_score(y_true, y_proba)  # "variance explained" on 0–1 outcome

    return {
        "accuracy": acc,
        "auc": auc,
        "log_loss": ll,
        "mae": mae,
        "r2": r2
    }
    
dict_results = []
    
# ============================================================================
# BASELINE: Elo Probability
# ============================================================================

print("\n" + "-"*80)
print("BASELINE: ELO PROBABILITY")
print("-"*80)

if 'player_win_prob_elo' in X_test.columns:
    elo_preds = (X_test['player_win_prob_elo'] > 0.5).astype(int)
    elo_probs = X_test['player_win_prob_elo']
    
    print(f"Accuracy: {accuracy_score(y_test, elo_preds):.4f}")
    print(f"ROC AUC: {roc_auc_score(y_test, elo_probs):.4f}")
    print("\nThis is your baseline to beat!")
    
elo_metrics = eval_classifier(y_test, elo_probs)
elo_metrics["model"] = "Baseline Elo"
print(elo_metrics)
if not any(d["model"] == elo_metrics["model"] for d in dict_results):
    dict_results.append(elo_metrics)


--------------------------------------------------------------------------------
BASELINE: ELO PROBABILITY
--------------------------------------------------------------------------------
Accuracy: 0.6392
ROC AUC: 0.7062

This is your baseline to beat!
{'accuracy': 0.6397989949748744, 'auc': 0.7062171561324209, 'log_loss': 0.6225268276429958, 'mae': 0.4282626837500031, 'r2': 0.1318078501639368, 'model': 'Baseline Elo'}


In [4]:
"""
Fix Multicollinearity - Remove Redundant Features
Run this AFTER preprocessing_for_modeling.py

Two strategies provided:
1. Keep DIFFERENCES, drop original pairs (RECOMMENDED)
2. Keep ORIGINALS, drop differences
"""

import pandas as pd
import numpy as np

print("="*80)
print("FIXING MULTICOLLINEARITY")
print("="*80)

print("\nPROBLEM: Perfect multicollinearity detected")
print("  - age_diff = player_age - opponent_age")
print("  - rank_diff = player_rank - opponent_rank")
print("  - elo_diff = player_elo - opponent_elo")
print("\nSOLUTION: Drop redundant features")

# ============================================================================
# STRATEGY 1: KEEP DIFFERENCES (RECOMMENDED)
# ============================================================================

print("\n" + "-"*80)
print("STRATEGY 1: KEEP DIFFERENCES, DROP ORIGINALS (RECOMMENDED)")
print("-"*80)

# Drop the original features, keep the differences
features_to_drop_v1 = [
    'player_age', 'opponent_age',      # Keep age_diff
    'player_rank', 'opponent_rank',    # Keep rank_diff
    'player_elo', 'opponent_elo',      # Keep elo_diff
    'player_form', 'opponent_form',  # 'player_form' feature also causes multicollinearity
    'player_height', 'opponent_height',
    'player_points', 'opponent_points',
    'player_rating', 'opponent_rating',
    'player_form', 'opponent_form',
    'rank_diff', 'elo_diff', 'form_diff',  # Drop raw differences
]

# Only drop if they exist
features_to_drop_v1 = [f for f in features_to_drop_v1 if f in X_train.columns]

X_train_fixed_v1 = X_train.drop(columns=features_to_drop_v1)
X_test_fixed_v1 = X_test.drop(columns=features_to_drop_v1)

X_train_scaled_fixed_v1 = X_train_scaled.drop(columns=features_to_drop_v1)
X_test_scaled_fixed_v1 = X_test_scaled.drop(columns=features_to_drop_v1)

print(f"\nDropped {len(features_to_drop_v1)} features: {features_to_drop_v1}")
print(f"New shape: {X_train_fixed_v1.shape}")


"""
# ============================================================================
# STRATEGY 2: KEEP ORIGINALS (ALTERNATIVE)
# ============================================================================

print("\n" + "-"*80)
print("STRATEGY 2: KEEP ORIGINALS, DROP DIFFERENCES (ALTERNATIVE)")
print("-"*80)

# Drop the engineered differences, keep original features
features_to_drop_v2 = [
    'age_diff',      # Keep player_age, opponent_age
    'rank_diff',     # Keep player_rank, opponent_rank
    'elo_diff'       # Keep player_elo, opponent_elo
]

features_to_drop_v2 = [f for f in features_to_drop_v2 if f in X_train.columns]

X_train_fixed_v2 = X_train.drop(columns=features_to_drop_v2)
X_test_fixed_v2 = X_test.drop(columns=features_to_drop_v2)

X_train_scaled_fixed_v2 = X_train_scaled.drop(columns=features_to_drop_v2)
X_test_scaled_fixed_v2 = X_test_scaled.drop(columns=features_to_drop_v2)

print(f"\nDropped {len(features_to_drop_v2)} features: {features_to_drop_v2}")
print(f"New shape: {X_train_fixed_v2.shape}")

# ============================================================================
# COMPARISON
# ============================================================================

print("\n" + "="*80)
print("WHICH STRATEGY TO USE?")
print("="*80)

print("\nSTRATEGY 1 (Keep Differences) - RECOMMENDED")
print("  Pros:")
print("    - More interpretable (difference directly shows advantage)")
print("    - Fewer features (simpler model)")
print("    - Captures relative strength directly")
print("  Cons:")
print("    - Loses absolute values (might matter for some contexts)")
print("  Use for: Most prediction tasks")

print("\nSTRATEGY 2 (Keep Originals)")
print("  Pros:")
print("    - Keeps absolute values")
print("    - Model learns the difference automatically")
print("  Cons:")
print("    - More features (more complex)")
print("    - Perfect multicollinearity between pairs")
print("  Use for: When absolute values matter")
"""

# ============================================================================
# RECOMMENDED: USE STRATEGY 1
# ============================================================================

print("\n" + "="*80)
print("USING STRATEGY 1 (RECOMMENDED)")
print("="*80)

# Set the recommended versions as the main variables
X_train_no_mc = X_train_fixed_v1
X_test_no_mc = X_test_fixed_v1
X_train_scaled_no_mc = X_train_scaled_fixed_v1
X_test_scaled_no_mc = X_test_scaled_fixed_v1

print(f"\nCreated new variables without multicollinearity:")
print(f"  X_train_no_mc: {X_train_no_mc.shape}")
print(f"  X_test_no_mc: {X_test_no_mc.shape}")
print(f"  X_train_scaled_no_mc: {X_train_scaled_no_mc.shape}")
print(f"  X_test_scaled_no_mc: {X_test_scaled_no_mc.shape}")

print("\n" + "="*80)
print("NOW RETRAIN YOUR MODEL")
print("="*80)

print("\nUse these variables instead of the original ones:")

from sklearn.linear_model import LogisticRegression

# Retrain without multicollinearity
lr_fixed = LogisticRegression(max_iter=1000, random_state=42)
lr_fixed.fit(X_train_scaled_no_mc, y_train)

y_pred_fixed = lr_fixed.predict(X_test_scaled_no_mc)
y_prob_fixed = lr_fixed.predict_proba(X_test_scaled_no_mc)[:, 1]

from sklearn.metrics import accuracy_score, roc_auc_score
print(f"Accuracy: {accuracy_score(y_test, y_pred_fixed):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test, y_prob_fixed):.4f}")

lr_no_mc_metrics = eval_classifier(y_test, y_prob_fixed)
lr_no_mc_metrics["model"] = "Logistic Regression (No Multicollinearity)"
print(lr_no_mc_metrics)
if not any(d["model"] == lr_no_mc_metrics["model"] for d in dict_results):
    dict_results.append(lr_no_mc_metrics)

"""
print("\nOR use regularization with the original data:")

# Ridge (L2 regularization) - handles multicollinearity
lr_ridge = LogisticRegression(penalty='l2', C=1.0, max_iter=1000, random_state=42)
lr_ridge.fit(X_train_scaled, y_train)

# Lasso (L1 regularization) - does feature selection
lr_lasso = LogisticRegression(penalty='l1', solver='saga', C=1.0, max_iter=1000, random_state=42)
lr_lasso.fit(X_train_scaled, y_train)

"""

print("\n" + "="*80)
print("DONE")
print("="*80)


FIXING MULTICOLLINEARITY

PROBLEM: Perfect multicollinearity detected
  - age_diff = player_age - opponent_age
  - rank_diff = player_rank - opponent_rank
  - elo_diff = player_elo - opponent_elo

SOLUTION: Drop redundant features

--------------------------------------------------------------------------------
STRATEGY 1: KEEP DIFFERENCES, DROP ORIGINALS (RECOMMENDED)
--------------------------------------------------------------------------------

Dropped 19 features: ['player_age', 'opponent_age', 'player_rank', 'opponent_rank', 'player_elo', 'opponent_elo', 'player_form', 'opponent_form', 'player_height', 'opponent_height', 'player_points', 'opponent_points', 'player_rating', 'opponent_rating', 'player_form', 'opponent_form', 'rank_diff', 'elo_diff', 'form_diff']
New shape: (39800, 28)

USING STRATEGY 1 (RECOMMENDED)

Created new variables without multicollinearity:
  X_train_no_mc: (39800, 28)
  X_test_no_mc: (9950, 28)
  X_train_scaled_no_mc: (39800, 28)
  X_test_scaled_no_mc: (9

In [7]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

train_idx, test_idx = next(gss.split(X, y, groups=X['match_id']))

X_train, X_test = X.iloc[train_idx].drop('match_id',axis=1), X.iloc[test_idx].drop('match_id',axis=1)
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

In [9]:
X_train

,Best of,player_rank,player_points,player_elo,player_h2h,opponent_rank,opponent_points,opponent_elo,opponent_h2h,player_height,...,opponent_hand_Unknown,rank_diff,elo_diff,player_h2h_rate,rating_diff,height_diff,points_diff,age_diff,proba_rank_200,proba_form_50
0,3.0,31.0,1195.0,2161.852551,0,72.0,691.0,1880.594310,0,188.0,...,False,-41.0,281.258241,0.50,-0.383,8.0,504.0,8.2,0.615864,0.500000
1,3.0,55.0,790.0,1997.849469,0,79.0,621.0,1964.068655,0,191.0,...,True,-24.0,33.780814,0.50,-0.527,3.0,169.0,2.4,0.568641,0.500000
2,3.0,80.0,620.0,1741.733072,0,95.0,557.0,1734.294841,0,191.0,...,False,-15.0,7.438231,0.50,-0.850,3.0,63.0,7.8,0.543066,0.500000
3,3.0,40.0,973.0,1916.842418,0,52.0,810.0,2152.516157,0,178.0,...,False,-12.0,-235.673739,0.50,-0.001,-5.0,163.0,1.8,0.534484,0.500000
4,3.0,289.0,168.0,1875.547138,0,89.0,575.0,1867.531447,0,180.0,...,False,200.0,8.015690,0.50,0.000,-3.0,-407.0,0.8,0.090909,0.500000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49745,3.0,1.0,10330.0,1500.000000,2,7.0,3855.0,1500.000000,0,191.0,...,False,-6.0,0.000000,1.00,0.527,8.0,6475.0,-1.9,0.517263,0.505331
49746,3.0,2.0,7315.0,1500.000000,4,5.0,4300.0,1500.000000,4,198.0,...,False,-3.0,0.000000,0.50,0.227,2.0,3015.0,1.3,0.508634,0.506361
49747,3.0,7.0,3855.0,1500.000000,0,1.0,10330.0,1500.000000,2,183.0,...,False,6.0,0.000000,0.00,-0.527,-8.0,-6475.0,1.9,0.482737,0.494669
49748,3.0,1.0,10330.0,1500.000000,3,5.0,4300.0,1500.000000,1,191.0,...,False,-4.0,0.000000,0.75,1.547,-5.0,6030.0,-3.0,0.511511,0.494433


In [ ]:
# ============================================================================
# MODEL 1: LOGISTIC REGRESSION (use scaled data)
# ============================================================================

print("\n" + "-"*80)
print("MODEL 1: LOGISTIC REGRESSION")
print("-"*80)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test, y_prob_lr):.4f}")

lr_metrics = eval_classifier(y_test, y_prob_lr)
lr_metrics["model"] = "Logistic Regression"
print(lr_metrics)

## Random forest

In [28]:
# ============================================================================
# MODEL 2: RANDOM FOREST (use original scale)
# ============================================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import ParameterGrid
print("\n" + "-"*80)
print("MODEL 2: RANDOM FOREST")
print("-"*80)

rf_param_grid = {
    'n_estimators': [500, 800, 1000],
    'max_depth': [None, 10, 50,100],
    'min_samples_leaf': [3, 4, 5],
    'min_samples_split': [10, 15, 20],
}


best_rf = None
best_logloss = np.inf
best_params = None

for params in ParameterGrid(rf_param_grid):
    model = RandomForestClassifier(**params,random_state=42,n_jobs=-1)
    model.fit(X_train, y_train)
    proba_val = model.predict_proba(X_test)[:, 1]
    ll = log_loss(y_test, proba_val)
    if ll < best_logloss:
        best_logloss = ll
        best_rf = model
        best_params = params

print("Best RandomForest params:", best_params)
rf_proba = best_rf.predict_proba(X_test)[:, 1]
rf_metrics = eval_classifier(y_test, rf_proba)
rf_metrics["model"] = "RandomForest"
print(rf_metrics)
#if not any(d["model"] == "CatBoost" for d in dict_results):
    #dict_results.append(cat_metrics)




y_pred_rf = best_rf.predict(X_test)
y_prob_rf = best_rf.predict_proba(X_test)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test, y_prob_rf):.4f}")


--------------------------------------------------------------------------------
MODEL 2: RANDOM FOREST
--------------------------------------------------------------------------------
Best RandomForest params: {'max_depth': None, 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 1000}
{'accuracy': 0.724321608040201, 'auc': 0.7933395621322693, 'log_loss': 0.5491638763761104, 'mae': 0.3763048865313833, 'r2': 0.2578141057703417, 'model': 'RandomForest'}
Accuracy: 0.7243
ROC AUC: 0.7933


In [4]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import randint

param_dist = {
    'n_estimators': randint(200, 1000),
    'max_depth': randint(5, 50),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 20),
    'max_features': ['sqrt', 'log2', None],
}
rf = RandomForestClassifier(random_state=42)
rs = RandomizedSearchCV(rf, param_dist, n_iter=30, cv=5, scoring='accuracy', n_jobs=-1)
rs.fit(X_train, y_train)

print(rs.best_params_)
print(rs.best_score_)


KeyboardInterrupt: 

In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from pathlib import Path

# ============================================================================
# LOAD DATA
# ============================================================================

# Choose your version
# Get script location and build path from there
script_dir = Path.cwd()
data_path = script_dir / '..' / 'data' / 'processed' / 'model_ready' / 'features_no_ytd.csv'

print("Loading from:", data_path.resolve())
df = pd.read_csv(data_path)

print(f"Original shape: {df.shape}")
print(f"Target distribution:\n{df['won'].value_counts()}\n")

# ============================================================================
# 1. DROP NON-PREDICTIVE COLUMNS
# ============================================================================

drop_cols = [
    'player_name',      # Names not useful for prediction
    'opponent_name',
    'stats_url',        # Metadata
    'Date',             # Will use time-based split instead
    'ATP',              # Redundant with other context
    'Tournament',       # High cardinality, can cause overfitting
    'Location'          # High cardinality
]

# Keep only columns that exist
drop_cols = [col for col in drop_cols if col in df.columns]
df_model = df.drop(columns=drop_cols)

print(f"After dropping non-predictive columns: {df_model.shape}")

# ============================================================================
# 2. IDENTIFY COLUMN TYPES
# ============================================================================

# Target
target = 'won'

# Categorical columns
categorical_cols = [
    'Surface',          # Hard, Clay, Grass
    'Court',            # Indoor, Outdoor
    'Round',            # Round of 32, Finals, etc.
    'Series',           # ATP250, ATP500, etc.
    'player_hand',      # L, R
    'opponent_hand',    # L, R
]

# Keep only existing categorical columns
categorical_cols = [col for col in categorical_cols if col in df_model.columns]

# Numerical columns (everything else except target)
numerical_cols = [col for col in df_model.columns 
                  if col not in categorical_cols and col != target]

print(f"\nCategorical columns ({len(categorical_cols)}): {categorical_cols}")
print(f"Numerical columns ({len(numerical_cols)}): {len(numerical_cols)} columns")

# ============================================================================
# 3. HANDLE MISSING VALUES
# ============================================================================

print("\n" + "="*80)
print("MISSING VALUES")
print("="*80)
df_model['player_hand'] = df['player_hand'].replace('U','Unknown')
df_model['opponent_hand'] = df['opponent_hand'].replace('U','Unknown')
df_model['Round'] = df['Round'].replace('Round Robin','4th Round')
df_model['Series'] = df['Series'].replace('Masters Cup','ATP500')

missing_summary = df_model.isnull().sum()
missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)
print(f"\nColumns with missing values: {len(missing_summary)}")
for col, count in missing_summary.head(10).items():
    pct = count / len(df_model) * 100
    print(f"  {col:30s}: {count:6d} ({pct:5.1f}%)")

# Strategy for numerical columns
print("\nFilling numerical missing values with median...")
for col in numerical_cols:
    if df_model[col].isnull().any():
        median_val = df_model[col].median()
        df_model[col] = df_model[col].fillna(median_val)

# Strategy for categorical columns
print("Filling categorical missing values with 'Unknown'...")
for col in categorical_cols:
    if df_model[col].isnull().any():
        df_model[col] = df_model[col].fillna('Unknown')

# Verify no missing values remain
remaining_missing = df_model.isnull().sum().sum()
print(f"\nRemaining missing values: {remaining_missing}")
assert remaining_missing == 0, "Still have missing values!"



# ============================================================================
# 5. FEATURE ENGINEERING (Optional but recommended)
# ============================================================================

print("\n" + "="*80)
print("FEATURE ENGINEERING")
print("="*80)

# Ranking difference (negative = player is lower ranked / worse)
if 'player_rank' in df_model.columns and 'opponent_rank' in df_model.columns:
    df_model['rank_diff'] = df_model['player_rank'] - df_model['opponent_rank']
    print("Created: rank_diff")

# Elo difference (positive = player has higher Elo / better)
if 'player_elo' in df_model.columns and 'opponent_elo' in df_model.columns:
    df_model['elo_diff'] = df_model['player_elo'] - df_model['opponent_elo']
    print("Created: elo_diff")

# H2H win rate (avoid division by zero)
if 'player_h2h' in df_model.columns and 'opponent_h2h' in df_model.columns:
    total_h2h = df_model['player_h2h'] + df_model['opponent_h2h']
    df_model['player_h2h_rate'] = np.where(
        total_h2h > 0,
        df_model['player_h2h'] / total_h2h,
        0.5  # No history = 50/50
    )
    print("Created: player_h2h_rate")
    
# Rating difference
if 'player_rating' in df_model.columns and 'opponent_rating' in df_model.columns:
    df_model['rating_diff'] = df_model['player_rating'] - df_model['opponent_rating']
    print("Created: rating_diff")
    
# Height difference
if 'player_height' in df_model.columns and 'opponent_height' in df_model.columns:
    df_model['height_diff'] = df_model['player_height'] - df_model['opponent_height']
    print("Created: height_diff")
    
# Point difference
if 'player_points' in df_model.columns and 'opponent_points' in df_model.columns:
    df_model['points_diff'] = df_model['player_points'] - df_model['opponent_points']
    print("Created: points_diff")

# Age difference
if 'player_age' in df_model.columns and 'opponent_age' in df_model.columns:
    df_model['age_diff'] = df_model['player_age'] - df_model['opponent_age']
    print("Created: age_diff")
    
# Try different scale factors
if 'player_rank' in df_model.columns and 'opponent_rank' in df_model.columns:
    df_model['proba_rank_200'] = 1 / (1 + 10 ** ((df_model['player_rank'] - df_model['opponent_rank']) / 200))
    print("Created: proba_rank_200")
    
# Different scale factor for form
if 'player_form' in df_model.columns and 'opponent_form' in df_model.columns:
    df_model['proba_form_50'] = 1 / (1 + 10 ** ((df_model['player_form'] - df_model['opponent_form']) / 50))
    print("Created: proba_form_50")

print(f"\nFinal shape with engineered features: {df_model.shape}")

# ============================================================================
# 6. SEPARATE FEATURES AND TARGET
# ============================================================================

X = df_model.drop(columns=[target])
y = df_model[target]

print("\n" + "="*80)
print("TRAIN-TEST SPLIT")
print("="*80)

# Option A: Random split (simple)
X_train_cat, X_test_cat, y_train_cat, y_test_cat = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain set: {X_train_cat.shape}")
print(f"Test set: {X_test_cat.shape}")
print(f"Train target distribution: {y_train_cat.value_counts().to_dict()}")
print(f"Test target distribution: {y_test_cat.value_counts().to_dict()}")


# ============================================================================
# 8. SUMMARY
# ============================================================================

print("\n" + "="*80)
print("PREPROCESSING COMPLETE")
print("="*80)

print(f"\nFinal dataset:")
print(f"  Total samples: {len(X):,}")
print(f"  Total features: {X.shape[1]:,}")
print(f"  Train samples: {len(X_train):,}")
print(f"  Test samples: {len(X_test):,}")
print(f"  Target balance: {y.mean():.1%} won")

print("\nReady for modeling!")
print("\nYou now have:")
print("  X_train, X_test, y_train, y_test - Original scale")
print("  X_train_scaled, X_test_scaled - Scaled features")
print("\nUse original scale for tree-based models (Random Forest, XGBoost)")
print("Use scaled features for linear models (Logistic Regression, SVM)")

Loading from: C:\Users\harry\Desktop\tennis_betting_efficiency-features\data\processed\model_ready\features_no_ytd.csv
Original shape: (49750, 32)
Target distribution:
won
1    24875
0    24875
Name: count, dtype: int64

After dropping non-predictive columns: (49750, 26)

Categorical columns (6): ['Surface', 'Court', 'Round', 'Series', 'player_hand', 'opponent_hand']
Numerical columns (19): 19 columns

MISSING VALUES

Columns with missing values: 16
  opponent_rating               :  12670 ( 25.5%)
  player_rating                 :  12670 ( 25.5%)
  player_height                 :   4020 (  8.1%)
  opponent_height               :   4020 (  8.1%)
  opponent_hand                 :   3996 (  8.0%)
  player_hand                   :   3996 (  8.0%)
  player_age                    :   3996 (  8.0%)
  opponent_age                  :   3996 (  8.0%)
  form_diff                     :   3270 (  6.6%)
  player_form                   :   1798 (  3.6%)

Filling numerical missing values with median.

In [9]:
from catboost import CatBoostClassifier

cat = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="Logloss",
    verbose=0,
    random_seed=42
)

cat_param_grid = {
    "depth": [2, 4],
    "learning_rate": [0.05, 0.1],
    "iterations": [300, 500]
}

from sklearn.model_selection import ParameterGrid

best_cat = None
best_logloss = np.inf
best_params = None

for params in ParameterGrid(cat_param_grid):
    model = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="Logloss",
        verbose=0,
        random_seed=42,cat_features=categorical_cols,
        **params
    )
    model.fit(X_train_cat, y_train)
    proba_val = model.predict_proba(X_test_cat)[:, 1]
    ll = log_loss(y_test, proba_val)
    if ll < best_logloss:
        best_logloss = ll
        best_cat = model
        best_params = params

print("Best CatBoost params:", best_params)
cat_proba = best_cat.predict_proba(X_test_cat)[:, 1]
cat_metrics = eval_classifier(y_test, cat_proba)
cat_metrics["model"] = "CatBoost"
print(cat_metrics)
if not any(d["model"] == "CatBoost" for d in dict_results):
    dict_results.append(cat_metrics)

Best CatBoost params: {'depth': 4, 'iterations': 500, 'learning_rate': 0.1}
{'accuracy': 0.7353768844221106, 'auc': 0.8081003206989723, 'log_loss': 0.5316826976727561, 'mae': 0.3553839268720378, 'r2': 0.28487786428045947, 'model': 'CatBoost'}
